In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
class model:
    def __init__(self,layers):
        self.layers = layers
        self.layer_length = len(self.layers)

        self.weights = []
        self.bias = []
        self._cache = []

        for i in range(self.layer_length - 1):
            w = np.random.randn(layers[i+1],layers[i]) * 1 / np.sqrt(layers[i])
            b = np.zeros(layers[i+1],1)

            self.weights.append(w)
            self.bias.append(b)

    def Relu(self,z):
        return np.maximum(0,z)
    
    def Relu_deriv(self,z):
        return (z>0).astype(float)
    
    def sigmoid(self,z):
        return 1/(1 + np.exp(-z))
    
    def softmax(self, z):
        exp_z = np.exp(z - np.max(z))
        
        return exp_z / np.sum(exp_z)
    
    @staticmethod
    def one_hot_encode(Y, num_classes=10):
        one_hot_Y = np.zeros((num_classes, Y.size))
        one_hot_Y[Y.astype(int), np.arange(Y.size)] = 1

        return one_hot_Y.astype(float)
    
    def forward(self,x):
        a_prev = x.T
        self.cache = []

        for i in range(len(self.weights)):
            z = self.weights[i] @ a_prev + self.bias[i]
            
            if(i == self.layer_length - 1):
                a_l = self.softmax(z)
            else:
                a_l = self.Relu(z)
                self.cache.append((z, a_prev))

            a_prev = a_l
    
        return a_prev

    def backprop(self,x,y,y_hat):
        M = y_hat.shape[1]
        L = self.weights.shape
        grads = {}

        dZ = y_hat - y.T

        for i in reversed(range(L)):
            if( i == L-1):
                z_curr,a_prev = self._cache[i-1]
            else:
                z_curr,a_prev = self._cache[i]

            w_curr = self.weights

            grads['dW' + str(i)] = (1 / M) * dZ @ a_prev.T
            grads['db' + str(i)] = (1 / M) * np.sum(dZ, axis=1, keepdims=True)

            if i != 0:
                dA = w_curr.T @ dZ
                dZ = dA * self.Relu_deriv(z_curr)

        return grads   

    def update_params(self,grad,learning_rate):
        for i in range(len(self.weights)):
            self.weights[i] -= learning_rate * grad['dW' + str(i)] 
            self.bias[i] -= learning_rate * grad['db' + str(i)]



In [ ]:
def train(model,x_train,y_train_oh,learning_rate,epoch):
    M = len(x_train[0])

    for epochs in range(epoch):
        y_hat = model.forward(x_train)

        loss = - (1 / M) * np.sum(y_train_oh * np.log(y_hat.T))

        grads = model.backprop(y_hat, y_train_oh)   

        model.update_params(grads,learning_rate)

        if (epochs + 1) % 10 == 0:
            # Calculate accuracy to see performance
            predictions = np.argmax(y_hat, axis=0) # Get the index of the highest probability
            actual_labels = np.argmax(y_train_oh.T, axis=1) # Get the original class index
            accuracy = np.mean(predictions == actual_labels) * 100
            
            print(f"Epoch {epochs + 1}/{epochs} | Loss: {loss:.4f} | Accuracy: {accuracy:.2f}%")